# Week 4 Mini Project: EdTech Student Success Prediction & Learning Behavior Analytics Dashboard
**Course**: MACSE502 - Programming for Data Science Lab  
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Date**: 30-07-2026 | **Faculty**: Dr. Rajasekhara Babu M  

---
## 1. Executive Summary & Problem Formulation
Higher education institutions and EdTech platforms require early-warning systems to anticipate student academic attrition. By analyzing digital trace data—such as attendance rates, LMS clicks, assignment submissions, and quiz scores—we can intervene before dropout occurs. This project implements feature transformation (logarithmic transform), categorical discretization, correlation matrices, and violin plot distributions. The **Mini Project Extension** engineers a **Composite Learner Engagement Index (CLEI)**, fits a predictive logistic completion probability model, segments the student cohort into risk tiers, and triggers automated personalized academic prescriptions.

### Objectives
1. Construct a 100-student learner engagement DataFrame containing study hours, quiz scores, assignment grades, and completion records.
2. Engineer `Log_StudyHours`, categorize `QuizScore` into performance tiers, and compute Pearson correlation matrices.
3. Render advanced violin plots comparing score distributions between completers and non-completers.
4. Implement the **Mini Project Extension**: CLEI score formulation, logistic success curve modeling, retention risk triage, and prescriptive learning intervention engine.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 15)
print('Pandas & Seaborn loaded.')

## 2. Ingestion of Learner Behavioral Telemetry

In [ ]:
csv_path = '../data/w4_edtech_learner_engagement.csv'
if not os.path.exists(csv_path):
    csv_path = 'data/w4_edtech_learner_engagement.csv'
df = pd.read_csv(csv_path)
print(f'Ingested cohort records: {df.shape[0]} students with {df.shape[1]} attributes.')
display(df.head())

## 3. Core Feature Engineering & EDA Operations

In [ ]:
# Step 1: Log transformation of StudyHours
df['Log_StudyHours'] = np.log1p(df['StudyHours']).round(4)

# Step 2: Categorize QuizScore
bins = [0, 55, 75, 100]
labels = ['Needs Support', 'Progressing', 'Excelling']
df['QuizCategory'] = pd.cut(df['QuizScore'], bins=bins, labels=labels, right=False)
print('Quiz Performance Tier Distribution:')
print(df['QuizCategory'].value_counts())

# Step 3: Correlation Matrix
num_cols = ['StudyHours', 'Log_StudyHours', 'QuizScore', 'AssignmentScore', 'AttendanceRate', 'InteractionCount', 'CourseCompletion']
corr_mat = df[num_cols].corr()
display(corr_mat.round(3))

# Feature Ranking with Completion
print('\nCorrelation with Course Completion:')
print(corr_mat['CourseCompletion'].drop('CourseCompletion').sort_values(ascending=False).round(3))

## 4. Mini Project Extension: Predictive Success Modeling & Prescriptive Engine

In [ ]:
# 1. Composite Learner Engagement Index (CLEI)
norm_sh = (df['StudyHours'] - df['StudyHours'].min()) / (df['StudyHours'].max() - df['StudyHours'].min())
norm_att = (df['AttendanceRate'] - df['AttendanceRate'].min()) / (df['AttendanceRate'].max() - df['AttendanceRate'].min())
norm_quiz = (df['QuizScore'] - df['QuizScore'].min()) / (df['QuizScore'].max() - df['QuizScore'].min())
norm_act = (df['InteractionCount'] - df['InteractionCount'].min()) / (df['InteractionCount'].max() - df['InteractionCount'].min())

df['CLEI_Score'] = (0.30 * norm_sh + 0.30 * norm_quiz + 0.25 * norm_att + 0.15 * norm_act) * 100

# 2. Logistic Success Probability
z = (df['CLEI_Score'] - 48.0) / 10.0
df['PredictedCompletionProb'] = 1 / (1 + np.exp(-z))

def risk_segment(p):
    if p < 0.40: return 'High Risk (Critical Intervention)'
    elif p < 0.70: return 'Moderate Risk (Needs Monitoring)'
    else: return 'Low Risk (On Track for Distinction)'
df['RiskTier'] = df['PredictedCompletionProb'].apply(risk_segment)
print('Cohort Risk Segmentation:')
print(df['RiskTier'].value_counts())

# 3. Prescriptive Learning Engine
def prescribe(row):
    recs = []
    if row['QuizScore'] < 55: recs.append('Enroll in Remedial Problem-Solving Labs')
    if row['StudyHours'] < 10: recs.append('Increase weekly study time by 3.5 hrs')
    if row['AttendanceRate'] < 75: recs.append('Mandatory mentor check-in for attendance remediation')
    if row['InteractionCount'] < 20: recs.append('Participate in peer discussion forum threads')
    if not recs: recs.append('Eligible for Advanced Honors Projects & Peer Mentoring')
    return ' | '.join(recs)
df['PrescriptiveAction'] = df.apply(prescribe, axis=1)
display(df[['LearnerID', 'QuizScore', 'AttendanceRate', 'CLEI_Score', 'PredictedCompletionProb', 'RiskTier', 'PrescriptiveAction']].head(6))

## 5. Visual Learning Analytics Cockpit

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('EdTech Student Success Prediction & Learning Behavior Analytics Dashboard\nVEDANT NIMKAR (26MML0045) | Week 4 Mini Project', fontsize=15, weight='bold', y=0.98)

# Subplot 1: Correlation Heatmap
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=1, ax=axes[0, 0])
axes[0, 0].set_title('Inter-Feature Correlation Matrix', weight='bold')

# Subplot 2: Violin Plot of QuizScore by Completion
sns.violinplot(data=df, x='CourseCompletion', y='QuizScore', palette={0: '#d9534f', 1: '#5cb85c'}, inner='quartile', ax=axes[0, 1])
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(['Incomplete (0)', 'Completed (1)'])
axes[0, 1].set_title('Violin Distribution of Quiz Scores by Completion Status', weight='bold')
axes[0, 1].set_xlabel('Course Completion Status')
axes[0, 1].set_ylabel('Quiz Score (0 - 100)')

# Subplot 3: Logistic Completion Probability Curve
risk_colors = {'High Risk (Critical Intervention)': '#d9534f', 'Moderate Risk (Needs Monitoring)': '#f0ad4e', 'Low Risk (On Track for Distinction)': '#5cb85c'}
for tier, grp in df.groupby('RiskTier'):
    axes[1, 0].scatter(grp['CLEI_Score'], grp['PredictedCompletionProb'], label=tier, color=risk_colors[tier], s=70, edgecolor='black', alpha=0.85)
x_synth = np.linspace(df['CLEI_Score'].min(), df['CLEI_Score'].max(), 200)
z_synth = (x_synth - 48.0) / 10.0
p_synth = 1 / (1 + np.exp(-z_synth))
axes[1, 0].plot(x_synth, p_synth, color='#1d3557', linestyle='--', linewidth=2, label='Logistic Decision Curve')
axes[1, 0].axhline(0.50, color='gray', linestyle=':', label='Threshold (0.50)')
axes[1, 0].set_title('Predicted Completion Probability vs. Composite Engagement (CLEI)', weight='bold')
axes[1, 0].set_xlabel('Composite Learner Engagement Index (CLEI 0-100)')
axes[1, 0].set_ylabel('Predicted Probability of Completion')
axes[1, 0].legend(loc='lower right', frameon=True, fontsize=9)

# Subplot 4: Risk Distribution Pie Chart
tier_counts = df['RiskTier'].value_counts()
axes[1, 1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', colors=[risk_colors[t] for t in tier_counts.index], startangle=140, pctdistance=0.75, wedgeprops=dict(width=0.45, edgecolor='black', linewidth=1.5))
axes[1, 1].set_title('Cohort Retention Risk Segmentation Distribution', weight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()